In [ ]:
# 16 — Parse the handyman CV PDFs (unstructured → text)
# ai_parse_document over every PDF in the CV Volume, in chunks so progress is saved as it goes; only files
# not yet parsed successfully are processed, so re-runs are incremental. The parsed text replaces
# handyman_details.cv_raw_text in Lakebase.

In [ ]:
%pip install -q "psycopg[binary]>=3.1"

In [ ]:
try:
    dbutils.library.restartPython()
except NameError:
    pass

In [ ]:
import os, sys, time
sys.path.insert(0, os.getcwd())          # pipeline_lib.py lives next to this notebook
import pipeline_lib as pl
from pyspark.sql import functions as F

dbutils.widgets.text("run_id", "manual")
RUN_ID = dbutils.widgets.get("run_id")
pl.ensure_tables(spark)

import psycopg
PG_URL = dbutils.secrets.get("maintops", "lakebase_pg_url")
CHUNK = 2500        # PDFs per saved batch: progress survives a crash
PARALLELISM = 64    # partitions per batch, so many PDFs are parsed at the same time
spark.sql(f"""CREATE TABLE IF NOT EXISTS {pl.T_CV_PARSED} (
                path STRING, user_id BIGINT, parsed_text STRING, error STRING, parsed_at TIMESTAMP)
              COMMENT 'ai_parse_document output per handyman CV PDF'""")

In [ ]:
started = time.time()
all_paths = [f.path.replace("dbfs:", "") for f in dbutils.fs.ls(pl.CV_DIR) if f.path.endswith(".pdf")]
done = {r.path for r in spark.table(pl.T_CV_PARSED).where("error IS NULL").select("path").collect()}
todo = [p for p in all_paths if p not in done]
print(f"{len(all_paths):,} PDFs, {len(done):,} already parsed, {len(todo):,} to parse")

for i in range(0, len(todo), CHUNK):
    chunk = todo[i:i + CHUNK]
    parsed = (spark.read.format("binaryFile").load(chunk)
              .repartition(PARALLELISM)
              .select(F.regexp_replace("path", "^dbfs:", "").alias("path"),
                      F.expr("ai_parse_document(content, map('version', '2.0'))").alias("doc"))
              .select("path",
                      F.expr("concat_ws('\\n\\n', transform(try_cast(doc:document:elements AS ARRAY<VARIANT>), "
                             "e -> try_cast(e:content AS STRING)))").alias("parsed_text"),
                      F.expr("try_cast(doc:error_status AS STRING)").alias("error"))
              # CV files are named cv_<user_id>.pdf, so no join is needed to find the owner
              .withColumn("user_id", F.regexp_extract("path", r"cv_(\d+)\.pdf$", 1).cast("bigint"))
              .withColumn("parsed_at", F.current_timestamp()))
    parsed.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(pl.fq("_stage_cv_parsed"))
    spark.sql(f"""MERGE INTO {pl.T_CV_PARSED} t USING {pl.fq('_stage_cv_parsed')} s ON t.path = s.path
                  WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
    print(f"  parsed {min(i + CHUNK, len(todo)):,}/{len(todo):,}")

In [ ]:
# Replace the generator's CV text in Lakebase with the parsed text
cv = spark.table(pl.T_CV_PARSED).where("error IS NULL AND user_id IS NOT NULL AND length(parsed_text) > 0")
with psycopg.connect(PG_URL) as conn, conn.cursor() as cur:
    cur.execute("CREATE TEMP TABLE _cv (user_id bigint, cv_raw_text text) ON COMMIT DROP")
    with cur.copy("COPY _cv (user_id, cv_raw_text) FROM STDIN") as cp:
        for row in cv.select("user_id", "parsed_text").toLocalIterator():
            cp.write_row(tuple(row))
    cur.execute("""UPDATE maintops.handyman_details d SET cv_raw_text = c.cv_raw_text FROM _cv c
                   WHERE d.user_id = c.user_id AND d.cv_raw_text IS DISTINCT FROM c.cv_raw_text""")
    updated = cur.rowcount
    conn.commit()

stats = spark.table(pl.T_CV_PARSED).agg(F.count("*").alias("n"), F.count_if(F.col("error").isNull()).alias("ok"),
                                        F.avg(F.length("parsed_text")).alias("avg_len")).first()
print(f"cv_parsed: {stats.n:,} rows, {stats.ok:,} ok, avg {stats.avg_len:.0f} chars; Lakebase rows updated: {updated:,}")
pl.log_step(spark, RUN_ID, "16_parse_cvs", started, rows_in=len(todo), rows_out=stats.ok,
            rows_rejected=stats.n - stats.ok,
            checks={"all_pdfs_parsed": stats.ok >= 0.99 * len(all_paths),
                    "text_extracted": (stats.avg_len or 0) > 500})